**GUIDE**

The goal of `fastllm` library which we are currently refactoring module by module here in these dialogs is to create a unified llm api for OpenAI responses, OpenAI chat, Anthropic and Gemini apis. Users should be able to use this unified / canonical library to be able to swap models/providers effortlessly without needing to change any code.

Refactored dialogs so far:

- `aai-ws/fastllm/nbs/00_types` exists is to create a canonical internal representations across model providers so that fastllm can be a unified api where users can swap models without changing code during a chat session. 
- `aai-ws/fastllm/nbs/01_normalize` exists to convert model responses or stream events into normalized internal representations.
- `aai-ws/fastllm/nbs/02_streaming` provides lossless stream collation (`acollect_stream`) that collects fragmented Delta streams into `StreamSummary` with a `Msg` matching the shape of `Completion.message`. Covers tool call assembly, thinking/text interleaving, and server tool result preservation across all four providers.


You can find the pre-refactor and fully functional snapshot of this library `~/aai-ws/fastllm/fastllm_py/*.py`. The already refactored modules might differ from their counterparts in `fastllm_py/types.py` and `fastllm_py/normalize.py`, as refactoring involves simplifying code, refactoring the code to fastai / literate programming style, redesigning parts, and fixing issues/bugs.


 When users continue the chat with the same provider raw data is used for constructing the provider native input with all required fields to keep the cache intact. When switching over to providers, canonical data is transformed into provider counterpart with best effort.

## Changes Summary (so far)

*This pinned note serves as a persistent reference for SolveitAI context, since earlier prompts and tool results get truncated in chat history. It captures all changes made so far so we can continue without losing track.*

### Goal
Align `StreamSummary` with `Completion` so both produce consistent `Msg` objects with properly ordered `Part`s, and add canonical thinking/reasoning support across all providers.

### 1. `Delta` type (`00_types`)
- Added `thinking: str = ""` field — parallel to `text`, so streaming consumers distinguish reasoning from output text
- Added `model` and `provider` fields

### 2. `StreamSummary` (`02_streaming`)
- Added `model: str` and `provider: str` fields
- Added `message: Msg` field — built from collated text + thinking + tool_calls, mirroring `Completion.message`
- Removed `text` as a top-level field (it's now inside `message.content` as parts)

### 3. `acollect_stream` (`02_streaming`) — order-preserving part assembly
- Builds `parts` list in true stream order using `_append_part` (merges consecutive same-type text/thinking) and `_reserve_tool` (inserts `None` placeholders for tool_use at first-appearance position)
- After stream ends, placeholders are filled with finalized `Part(type="tool_use", data=dict(id, name, arguments, **extra))`
- Handles interleaved thinking/text correctly (each type-switch starts a new Part)
- Now yields `{'text': chunk}` or `{'thinking': chunk}` dicts for incremental streaming
- Final yield is the completed `StreamSummary`

### 4. `ToolCall.extra` preservation (`02_streaming`)
- `_ToolBuf` now has `extra: dict` field
- `acollect_stream` merges `tc.extra` into buffer during streaming
- `_final_tool_calls` passes `extra=tb.extra` through to final `ToolCall` objects
- Tool_use `Part.data` spreads `**tc.extra` so shape matches completion normalizers

### 5. Stream event normalizers (`01_normalize`) — `Delta(thinking=...)`
- **Anthropic**: `thinking_delta` → `Delta(thinking=...)` instead of `Delta(text=...)`
- **OpenAI Responses**: Added `response.reasoning_text.delta` → `Delta(thinking=...)`
- **OpenAI Chat**: Check `reasoning_content` first → thinking-only Delta; otherwise text Delta
- **Gemini**: Check `thought: true` parts first → thinking-only Delta; otherwise text Delta

### 6. Completion normalizers (`01_normalize`) — `Part(type='thinking')`
- **Anthropic**: Already worked via `_ANT_PART` mapping
- **OpenAI Responses**: `_openai_responses_parts` handles `reasoning` output items → `Part(type='thinking')` from summary blocks
- **OpenAI Chat**: `normalize_openai_chat_completion` extracts `reasoning_content` → prepends `Part(type='thinking')`
- **Gemini**: `_gem_part_type` + `normalize_gemini_generate` exported — maps `thought: true` → `Part(type='thinking')`

### 7. Server tool result support (`01_normalize`, `02_streaming`)
- **`_ant_part_type`** (`01_normalize`): `*_tool_result` types now map to `'server_tool_result'` instead of `'tool_result'` — completions produce `Part(type='server_tool_result', data=<raw content block>)`
- **`acollect_stream`** (`02_streaming`): detects `content_block_start` events ending in `_tool_result` and appends `Part(type='server_tool_result', data=<raw content block>)` — streaming now preserves server tool results (e.g. `web_search_tool_result` with encrypted content needed for multi-turn chat)

### 8. Server/MCP tool call streaming fixes (`01_normalize`, `02_streaming`)
- **`_prime_tool_from_raw`** (`02_streaming`): broadened `content_block_start` check from `== "tool_use"` to `endswith("tool_use")` — `server_tool_use` and `mcp_tool_use` are now properly primed with `idx_to_key` mapping, fixing collation (was producing 2 separate tool calls instead of 1)
- **`normalize_anthropic_event`** (`01_normalize`): added `server=b.get("type")!="tool_use"` to ToolCall in `content_block_start` handler — streaming Deltas now carry `server=True` for server tools
- **`_ToolBuf`** (`02_streaming`): added `server: bool = False` field
- **`acollect_stream`** (`02_streaming`): propagates `tc.server` into `_ToolBuf` (sticky True)
- **`_final_tool_calls`** (`02_streaming`): passes `server=tb.server` through to final `ToolCall` objects

### Known gaps
- **Text `Part.data`**: Streaming text parts have `data=None` while completion text parts carry provider metadata (annotations, citations). Deferred — not needed for chat continuation.
- **`cache_creation`** missing in Anthropic streaming usage (present in completion usage but not streaming)


## Server Tool Results & Cross-Provider Web Search — Findings

### Anthropic Server Tools (web_search, web_fetch, etc.)

**Response structure:** In an assistant message, `server_tool_use` blocks are immediately followed by their `*_tool_result` blocks (e.g. `web_search_tool_result`) in the same `content` array. Text blocks with citations follow after. The `*_tool_result` content contains `encrypted_content` and `encrypted_index` blobs that must be passed back verbatim for multi-turn citation continuity.

**Streaming:** The `web_search_tool_result` arrives fully formed in a single `content_block_start` event — no subsequent deltas. Currently missing from our streaming path (not surfaced in `normalize_anthropic_event`).

**ID convention:** Server tool IDs use `srvtoolu_` prefix (vs `toolu_` for regular tools). Litellm uses this prefix as the heuristic for reconstruction.

### Anthropic Message Format Rules

1. Every `tool_use` block in an assistant message **must** have a corresponding `tool_result` in the next user message — no exceptions
2. `server_tool_use` + `*_tool_result` live together in the **assistant** content array (no user `tool_result` needed)
3. Anthropic does NOT distinguish between "web search" and "regular" tool_use by name — only by the `srvtoolu_` prefix and the presence of the result block

### Cross-Provider Swapping (OpenAI → Anthropic)

**Problem:** OpenAI treats web search as a regular tool (`web_search_call` with `ws_` prefixed id). When converting this history to Anthropic format, Anthropic sees a `tool_use` without a `tool_result` and returns 400.

**Solution (tested & working):** Convert the OpenAI web search tool_use into:
1. Assistant message: `[{"type": "tool_use", "id": "...", "name": "web_search", "input": {...}}]`
2. User message: `[{"type": "tool_result", "tool_use_id": "...", "content": "<extracted text>"}]`  
3. Assistant message: `[{"type": "text", "text": "<original response>"}]`

This works both with and without Anthropic's `web_search` tool enabled. Claude uses the tool result as context and responds naturally.

### Cross-Provider Swapping (Anthropic → Other)

When sending Anthropic history to OpenAI/Gemini, `server_tool_use` + `web_search_tool_result` blocks should be stripped or converted to plain text — other providers don't understand them. The text content with citations can be kept as-is (citations become inert text).

### Litellm Approach (reference)

- Stores `web_search_tool_result` blocks in `provider_specific_fields["web_search_results"]` on the message
- On reconstruction for Anthropic: checks `srvtoolu_` prefix → emits `server_tool_use` + matches result by `tool_use_id`
- Streaming: captures full `web_search_tool_result` from `content_block_start` event into accumulator
- `input_json_delta` events are only treated as tool call args when `current_content_block_type` is `tool_use` or `server_tool_use` (not for `*_tool_result`)

# Streaming

> Streaming helpers for lossless event collation.

In [ ]:
#| default_exp streaming

## Imports

In [ ]:
#| export
import json
from dataclasses import dataclass, field, fields
from fastcore.utils import *
from fastcore.meta import delegates
from fastspec.errors import *
from fastllm.types import *
from fastllm.normalize import *

In [ ]:
#| hide
import yaml,json
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms

In [ ]:
enable_cachy(doms=(*doms,'api.moonshot.ai'), hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = Path('../specs/')

ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.moonshot.ai/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
gem_cli.models

- models.generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a model response given an input `GenerateContentRequest`. Refer to the [text generation guide](https://ai.google.dev/gemini-api/docs/text-generation) for detailed usage information. Input capabilities differ between models, including tuned models. Refer to the [model guide](https://ai.google.dev/gemini-api/docs/models/gemini) and [tuning guide](https://ai.google.dev/gemini-api/docs/model-tuning) for details.*
- models.generate_answer(model, contents, answer_style, inline_passages, semantic_retriever, safety_settings, temperature): *Generates a grounded answer from the model given an input `GenerateAnswerRequest`.*
- models.stream_generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a [streamed response](https://ai.google.dev/gemini-api/docs/text-generation?lang=python#generate-a-text-stream) from the model given an input `GenerateContentRequest`.*
- models.embed_content(model, content, task_type, title, output_dimensionality): *Generates a text embedding vector from the input `Content` using the specified [Gemini Embedding model](https://ai.google.dev/gemini-api/docs/models/gemini#text-embedding).*
- models.batch_embed_contents(model, requests): *Generates multiple embedding vectors from the input `Content` which consists of a batch of strings represented as `EmbedContentRequest` objects.*
- models.count_tokens(model, contents, generate_content_request): *Runs a model's tokenizer on input `Content` and returns the token count. Refer to the [tokens guide](https://ai.google.dev/gemini-api/docs/tokens) to learn more about tokens.*
- models.batch_generate_content(model, batch): *Enqueues a batch of `GenerateContent` requests for batch processing.*
- models.async_batch_embed_content(model, batch): *Enqueues a batch of `EmbedContent` requests for batch processing. We have a `BatchEmbedContents` handler in `GenerativeService`, but it was synchronized. So we name this one to be `Async` to avoid confusion.*
- models.generate_message(model, prompt, temperature, candidate_count, top_p, top_k): *Generates a response from the model given an input `MessagePrompt`.*
- models.count_message_tokens(model, prompt): *Runs a model's tokenizer on a string and returns the token count.*
- models.get(name): *Gets information about a specific `Model` such as its version number, token limits, [parameters](https://ai.google.dev/gemini-api/docs/models/generative-models#model-parameters) and other metadata. Refer to the [Gemini models guide](https://ai.google.dev/gemini-api/docs/models/gemini) for detailed model information.*
- models.list(page_size, page_token): *Lists the [`Model`s](https://ai.google.dev/gemini-api/docs/models/gemini) available through the Gemini API.*
- models.predict(model, instances, parameters): *Performs a prediction request.*
- models.predict_long_running(model, instances, parameters): *Same as Predict but returns an LRO.*
- models.generate_text(model, prompt, temperature, candidate_count, max_output_tokens, top_p, top_k, safety_settings, stop_sequences): *Generates a response from the model given an input message.*
- models.embed_text(model, text): *Generates an embedding from the model given an input message.*
- models.batch_embed_text(model, texts, requests): *Generates multiple embeddings from the model given input text in a synchronous call.*
- models.count_text_tokens(model, prompt): *Runs a model's tokenizer on a text and returns the token count.*

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

### Delta

Canonical streaming event fragment

In [ ]:
#| export
@dataclass
class Delta:
    "Normalized streaming delta event."
    text: str = ""
    thinking: str = ""
    refusal: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)
    citations: list = field(default_factory=list)
    server_tool_result: dict = None
    finish_reason: str = None
    usage: Usage = None
    raw: dict = field(default_factory=dict)

In [ ]:
#| export
async def norm_and_yield(resp, norm_func):
    async for ev in resp:
        if (d := norm_func(ev)) is not None: yield d

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

`ToolCall` normalizes a tool call with `id`, `name`, and `arguments`, anything extra goes into `extra`

`tool_use` parts are built from tool calls, and store `ToolCall` fields in `data`, which will be converted into correct message type based on provider, and allow users to pass `Msg` directly to the chat history

### OpenAI Responses Events

OpenAI responses api returns collated responses in the last stream delta, so we can build the final `Completion` object using the `normalize_openai_response` function without needing anything else:

In [ ]:
#| export
def normalize_openai_response_event(ev, **kwargs):
    "Normalize OpenAI Responses API stream event into Delta."
    typ = ev.get("type")
    if typ == "response.output_text.delta":            return Delta(text=ev.get("delta"), raw=ev, **kwargs)
    if typ == "response.reasoning_text.delta":         return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.reasoning_summary_text.delta": return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.completed":                    return Delta(raw=ev, **kwargs)
    if typ == "error": raise api_error_from_event(ev)

In [ ]:
#| export
async def _acollect_stream_openai_responses(it, model=None, api_name=ApiName.openai, vendor_name='openai'):
    "Collect a Delta stream, yielding incremental chunks and a final normalized response."
    async for d in it:
        if d.text:     yield {'text': d.text}
        if d.thinking: yield {'thinking': d.thinking}
        t = d.raw.get('type')
        if t == 'response.failed': raise api_error_from_event(d.raw)
        if t in ('response.completed', 'response.incomplete'):
            yield normalize_openai_response(d.raw['response'], model, api_name=api_name, vendor_name=vendor_name)

#### Text

In [ ]:
mn,inp = 'gpt-4o-mini','Hi!'
resp = await oai_cli.responses.create_response(model=mn,input=inp)
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
# async for o in  acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in  _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

Hello

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hello! How can I assist you today?', data={'type': 'output_text', 'logprobs': [], 'text': 'Hello! How can I assist you today?', 'citations': []})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hello! How can I assist you today?', data={'type': 'output_text', 'logprobs': [], 'text': 'Hello! How can I assist you today?', 'citations': []})], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.usage, o.usage)
test_eq(comp.tool_calls, o.tool_calls)

#### Thinking

In [ ]:
mn,inp = 'o4-mini','What is 31231231 * 12312?'
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning={"effort": "low", "summary": "auto"})
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning={"effort": "low", "summary": "auto"},stream=True)
# async for o in acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

518

,

916

,

072

In [ ]:
for p in o.message.content: print(p,'\n')

Part(type=<PartType.thinking: 'thinking'>, text='**Calculating multiplication**\n\nThe user wants me to compute 31,231,231 multiplied by 12,312. I’ll break it down into smaller parts for easier calculations. First, I calculate 31,231,231 times 12,000, which I determine to be 374,774,772,000. Next, I calculate the multiplication with 312 separately and get 9,744,144,072. Adding these two results gives me a final answer of 384,518,916,072. To double-check, I can also use approximate methods to verify that my answer is close.', data={'id': 'rs_0c5e18b289c1e6c00069e7738ffd5081939fc820fec854fa9a', 'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': '**Calculating multiplication**\n\nThe user wants me to compute 31,231,231 multiplied by 12,312. I’ll break it down into smaller parts for easier calculations. First, I calculate 31,231,231 times 12,000, which I determine to be 374,774,772,000. Next, I calculate the multiplication with 312 separately and get 9,744,144,072. Adding th

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
# async for o in  acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='fc_094d0b47353783620069e77040fe9c81a0bd2653866ba3e8bb', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_pzmhOjJw6zBRudCXZYHsFE5C'}),
 ToolCall(id='fc_094d0b47353783620069e77040fea881a0acb7f00091b2ec71', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_Qzt76aAn9jvtfIWevBCRFIbH'})]

In [ ]:
o.tool_calls

[ToolCall(id='fc_01cf133e86e052160069e6365420288197b0d0c2e51d335198', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_6gRxeW6f3hTu6fcVwGYdn5qO'}),
 ToolCall(id='fc_01cf133e86e052160069e6365420388197b5dd7cb8556634d7', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_d0y28wT8Kl7x6jlwggb4xYHW'})]

In [ ]:
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_094d0b47353783620069e77040fe9c81a0bd2653866ba3e8bb', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_pzmhOjJw6zBRudCXZYHsFE5C'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_094d0b47353783620069e77040fea881a0acb7f00091b2ec71', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_Qzt76aAn9jvtfIWevBCRFIbH'})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_01cf133e86e052160069e6365420288197b0d0c2e51d335198', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_6gRxeW6f3hTu6fcVwGYdn5qO'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_01cf133e86e052160069e6365420388197b5dd7cb8556634d7', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_d0y28wT8Kl7x6jlwggb4xYHW'})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.usage, o.usage)

#### Web search

In [ ]:
mn,inp = 'gpt-4o-mini','What is the weather in Istanbul today?'
tools=[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = normalize_openai_response(resp, mn)

OpenAI responses text parts already include citations inline, but we still include it in `Part.data`:

In [ ]:
cites = comp.message.content[-1].data['citations']; cites

[{'type': 'url_citation',
  'end_index': 972,
  'start_index': 869,
  'title': 'April weather - Spring 2026 - Istanbul, Turkey',
  'url': 'https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai'}]

In [ ]:
Markdown(comp.message.content[-1].text)

<div class="prose">

As of 3:41 PM local time in Istanbul, the current weather is mostly cloudy with a temperature of 64°F (18°C).

## Weather for Hastane, Hadımköy-İstanbul Caddesi 34, 34555 Arnavutköy/İstanbul, Türkiye:
Current Conditions: Mostly cloudy, 64°F (18°C)

Hourly Forecast:
* 4:00 PM: 65°F (18°C), Cloudy
* 5:00 PM: 64°F (18°C), Cloudy
* 6:00 PM: 61°F (16°C), Showers
* 7:00 PM: 58°F (15°C), Cloudy
* 8:00 PM: 56°F (13°C), Cloudy
* 9:00 PM: 53°F (12°C), Cloudy
* 10:00 PM: 50°F (10°C), Cloudy
* 11:00 PM: 50°F (10°C), Cloudy
* 12:00 AM: 49°F (10°C), Cloudy
* 1:00 AM: 49°F (10°C), Rain
* 2:00 AM: 49°F (9°C), Rain
* 3:00 AM: 49°F (9°C), Cloudy


In April, Istanbul typically experiences average high temperatures around 59.5°F (15.3°C) and lows near 49.8°F (9.9°C). The city usually receives about 1.14 inches (29 mm) of rainfall over approximately 12.6 days during the month. ([weather-atlas.com](https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai))

Please note that weather conditions can change, so it's advisable to check for the latest updates. 

</div>

In [ ]:
comp.message.content[-1].text[cites[0]['start_index']:]

"([weather-atlas.com](https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai))\n\nPlease note that weather conditions can change, so it's advisable to check for the latest updates. "

Same with streaming:

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

As

 of

4

:

00

...

 for

 the

 latest

 updates

.

In [ ]:
cites = o.message.content[-1].data['citations']; cites

[{'type': 'url_citation',
  'end_index': 943,
  'start_index': 848,
  'title': 'Istanbul weather in April 2026 | Turkey: How hot?',
  'url': 'https://www.weather2travel.com/turkey/istanbul/april/?utm_source=openai'}]

In [ ]:
o.message.content[-1].text[cites[0]['start_index']:]

"([weather2travel.com](https://www.weather2travel.com/turkey/istanbul/april/?utm_source=openai))\n\nPlease note that weather conditions can change, so it's advisable to check for the latest updates. "

In [ ]:
comp.tool_calls

[ToolCall(id='ws_0e3ff0b4011e99140069e7705eb880819e85a94b7d9b7796c8', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})]

In [ ]:
o.tool_calls

[ToolCall(id='ws_090e0e74ee8537ed0069e774e330ac819fa0ebe682a82c3b7c', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})]

In [ ]:
comp.finish_reason, o.finish_reason

('stop', 'stop')

In [ ]:
comp.usage, o.usage

(Usage(prompt_tokens=309, completion_tokens=418, total_tokens=727, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 418, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 727, 'server_tool_use': {'web_search_call': 1}}),
 Usage(prompt_tokens=309, completion_tokens=403, total_tokens=712, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 403, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 712, 'server_tool_use': {'web_search_call': 1}}))

### OpenAI Chat Events

`PartAccum` part accumulator is used across openai chat, anthropic and gemini streaming events to collate the parts during parts.

In [ ]:
#| export
@dataclass
class PartAccum:
    parts: dict[Part|ToolCall] = field(default_factory=dict)
    tool_calls: list[ToolCall] = field(default_factory=list)
    
    def append(self, typ, index, txt='', citations=None, **tc_kwargs):
        'Create and accumulate same type sequential parts'
        if index not in self.parts: 
            if typ==PartType.tool_use: self.parts[index] = ToolCall(**tc_kwargs)
            else:                      self.parts[index] = Part(type=typ, text=txt, data=dict(citations=citations or []))
        else:
            if typ==PartType.tool_use:
                    new_args = tc_kwargs.get('arguments', '')
                    cur_args = self.parts[index].arguments
                    if isinstance(new_args, str) and isinstance(cur_args, str): self.parts[index].arguments += new_args
                    elif isinstance(new_args, str) and isinstance(cur_args, dict): self.parts[index].arguments = new_args
                    else: self.parts[index].arguments = new_args
            else: 
                self.parts[index].text += txt
                # anthropic citations have matching idx
                self.parts[index].data['citations'].extend(citations or [])
    
    def finalize(self, api_name=None):
        for idx,tc in self.parts.items():
            if isinstance(tc, ToolCall):
                if isinstance(tc.arguments, str): tc.arguments = json.loads(tc.arguments)
                self.tool_calls.append(tc)
                self.parts[idx] = Part(type=PartType.tool_use, data=dict(id=tc.id, name=tc.name, arguments=tc.arguments, server=tc.server, **tc.extra))
        
        # Don't merge for Anthropic to be able to keep text and citations grouped
        merged = []
        for p in self.parts.values():
            if api_name != ApiName.anthropic and merged and merged[-1].type == p.type and p.type in (PartType.text, PartType.thinking):
                merged[-1].text += p.text
                merged[-1].data['citations'].extend(p.data['citations'])
            else: merged.append(p)
        self.parts = merged
    
        # Broadcast: unlike anthropic gemini returns grounding metadata at the final event
        if api_name == ApiName.gemini:
            if citations:=self.parts[-1].data.get('citations'):
                for p in self.parts: 
                    if p.type == PartType.text: p.data['citations'] = citations

A generic `_acollect_stream` function that yields thinking and text (if needed we can yield tool calls to), and collates parts while keeping the order. A custom `index_fn` is used to resolve the index that the current `Delta` event belongs to.

In [ ]:
#| export
async def _acollect_stream(it, index_fn, model=None, api_name=None, vendor_name=None):
    "Collect a Delta stream, yielding incremental chunks and a final Completion."
    part_accum = PartAccum()
    deltas = []
    typ, last_typ, last_idx, last_idx = None, None, None, None
    async for d in it:
        if d.text:     
            yield {'text': d.text}
            typ = PartType.text
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.text)
        if d.thinking: 
            yield {'thinking': d.thinking}
            typ = PartType.thinking
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.thinking)
        if d.citations:
            yield {'citations': d.citations} # anthropic
            typ = PartType.text
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, citations=d.citations)
        for tc in d.tool_calls:
            args = tc.arguments.get('_delta', '') if '_delta' in tc.arguments else tc.arguments
            tc_kwargs = dict(id=tc.id, name=tc.name, arguments=args, server=tc.server, extra=tc.extra)
            typ = PartType.tool_use
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, **tc_kwargs)
        if d.server_tool_result: 
            typ = PartType.server_tool_result
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.parts[idx] = Part(type=typ, data=d.server_tool_result)
        if d.refusal:
            typ = PartType.refusal
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.refusal)        
        if d.finish_reason: fin = d.finish_reason
        if d.usage: usg = d.usage
        last_typ = typ
        deltas.append(d)
    part_accum.finalize(api_name)
    fin = canon_finish(fin, api_name, part_accum.tool_calls) # need to canon one more time with accum'd tool calls
    # tool calls and non-anthropic citations are yielded at the end
    yield Completion(d.raw.get('model', model),
            message=Msg(role="assistant", content=part_accum.parts),
            finish_reason=fin,
            usage=usg,
            tool_calls=part_accum.tool_calls,
            api_name=api_name,
            vendor_name=vendor_name,
            raw={'deltas':deltas})

In [ ]:
#| export
def openai_chat_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if d.tool_calls: 
        tc_idx = nested_idx(d.tool_calls, 0, 'extra', 'index')
        return f"tool_{tc_idx}", last_idx
    if not (last_typ or last_idx): return 0,0
    if typ == last_typ: return last_idx, last_idx
    return last_idx + 1, last_idx + 1

In [ ]:
#| export
_acollect_stream_openai_chat = partial(_acollect_stream, index_fn=openai_chat_index_fn, api_name=ApiName.openai_chat)

In [ ]:
#| export
def normalize_openai_chat_delta(ev, **kwargs):
    "Normalize a chat completion stream event."
    # usage always arrives as a single final event with choices: []
    if not (choices := ev.get("choices", [])): return Delta(usage=usage_from_openai(ev), raw=ev, **kwargs)
    # finish_reason arrives in its own dedicated chunk (empty delta, non-null finish_reason)
    if fin := nested_idx(ev, 'choices', 0, 'finish_reason'): return Delta(finish_reason=fin, raw=ev, **kwargs)
    # repurposed the common function
    tcs = openai_chat_tool_calls(ev, delta=True)
    dlt = nested_idx(choices, 0, 'delta')
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=dlt.get('content'), thinking=dlt.get('reasoning_content'), refusal=dlt.get('refusal'), tool_calls=tcs, raw=ev, **kwargs)

#### Text

In [ ]:
mn,msgs = 'gpt-4o-mini',[{"role": "user", "content": "Say hi"}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs)
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

Hi

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type='text', text='Hi there! How can I assist you today?', data={'citations': []})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hi! How can I assist you today?', data={'citations': []})], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

#### Thinking

OpenAI's Chat Completions API doesn't expose reasoning content — `reasoning_tokens` appear in usage but no `reasoning_content` field is returned. We use Kimi (`kimi-k2.5`) via Moonshot's OpenAI-compatible API to test thinking parts in chat completions streaming.

**NB**: Kimi's thinking is controlled via a `thinking` body param (not `reasoning_effort`). Use `_body={"thinking": {"type": "disabled"}}` to disable it, or `_body={"thinking": {"type": "enabled"}}` to enable. Since `thinking` isn't in the OpenAI spec, `fastspec` requires the `_body` escape hatch.

Kimi's thinking is binary — enabled (default) or disabled. There's no `reasoning_effort` level (low/medium/high) or `budget_tokens` like OpenAI/Anthropic.

**TODO**: Expose `_body`/`native` escape hatches to users in the high-level API so provider-specific params (like Kimi's `thinking`) can be passed through without needing to drop down to raw `fastspec` calls.

In [ ]:
mn,msgs = 'kimi-k2.5',[{"role": "user", "content": 'What is 31231231 * 12312?'}]
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,reasoning_effort='low')
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

,

916

,

072

**

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn = 'gpt-4o-mini'
msgs = [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch])
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch], stream=True, stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='call_3q4lLdRo9pG14Sffcka7aBBa', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
 ToolCall(id='call_W1xDfXoMw0bD3JztcCDrwIQ6', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})]

In [ ]:
o.tool_calls

[ToolCall(id='call_vvI8G2lh4NmvNR9EsGrdP7uU', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'index': 0, 'type': 'function'}),
 ToolCall(id='call_yrClxBDu5LXL5iZF93MNl85C', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'index': 1, 'type': 'function'})]

In [ ]:
comp.message.content

[Part(type='tool_use', text=None, data={'id': 'call_3q4lLdRo9pG14Sffcka7aBBa', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function'}),
 Part(type='tool_use', text=None, data={'id': 'call_W1xDfXoMw0bD3JztcCDrwIQ6', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function'})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_vvI8G2lh4NmvNR9EsGrdP7uU', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'index': 0, 'type': 'function'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_yrClxBDu5LXL5iZF93MNl85C', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'index': 1, 'type': 'function'})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

### Anthropic Messages Events

In [ ]:
#| export
def normalize_anthropic_event(ev, **kwargs):
    typ = ev.get("type")
    text, thinking, tcs, citations = None, None, [], None
    if typ == "content_block_start":
        cb = ev.get("content_block", {})
        if cb.get("type", "").endswith("_tool_result"): return Delta(server_tool_result=cb, raw=ev, **kwargs)
        if tc := anthropic_tool_call(cb): tcs = [tc]
    elif typ == "content_block_delta":
        d = ev.get("delta", {})
        dtyp = d.get("type")
        if   dtyp == "text_delta": text = d.get("text")
        elif dtyp == "thinking_delta": thinking = d.get("thinking")
        elif dtyp == "input_json_delta":
            tcs = [ToolCall(id=str(ev.get("index", "")), name="", arguments={"_delta": d.get("partial_json", '')})]
        elif dtyp == "citations_delta": 
            citations = listify(d.get('citation',[]))
    elif typ == "message_delta":
        fin = canon_finish(nested_idx(ev, 'delta', 'stop_reason'), 'anthropic')
        return Delta(finish_reason=fin, usage=usage_from_anthropic(ev), raw=ev, **kwargs)
    elif typ == "error": raise api_error_from_event(ev)
    return Delta(text=text, thinking=thinking, tool_calls=tcs, citations=citations, raw=ev, **kwargs)

In [ ]:
#| export
def anthropic_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    return nested_idx(d, 'raw', 'index'), None

In [ ]:
#| export
_acollect_stream_anthropic = partial(_acollect_stream, index_fn=anthropic_index_fn, api_name=ApiName.anthropic)

#### Thinking

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Say hi"}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, max_tokens=8000,
                                            thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=msgs,max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

Hi there! How are you doing today?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to say hi, which is a simple greeting request. I should respond in a friendly and welcoming manner.', data={'type': 'thinking', 'thinking': 'The user is asking me to say hi, which is a simple greeting request. I should respond in a friendly and welcoming manner.', 'signature': 'Er4CCmIIDBgCKkBlNieCYQXYfv8CZ9Le/gVCoMWG06rQBrlJ8iaa9mXoJzSovD3VlAcjMOyD3nSCYCpGEnJ2f6bqFtNy5yzO9+wQMhhjbGF1ZGUtc29ubmV0LTQtMjAyNTA1MTQ4ABIMMxtiJNjxC1AI4m2pGgy20Be1MqD10CvEcmsiMEkSeeczB4isM6+4mpkp/dJ8yJQaB9LpvNzePADcNsxq7VJFVGXLYb+VrHkNeoCiiSqJAVq63+ckbdMaDDs/3/hbMY2e/QSxJTc2vpEP4rxIiVeaGyUg0XYknERqAgIanM3HBBzgkp3mklfvcp6XnZZX0dS2+3084qRJUXPVR6IEvM+mIVjRq6bo9pt0WfauEbH8oHoklLvW+NrE+dUO9tb2Nx7aIYJJ2Gkic0ZMaxhbBKfKgLeIsE0YwBVaGAE='}), Part(type=<PartType.text: 'text'>, text='Hi there! How are you doing today? Is there anything I can help you with?', data={'type': 'text', 'text': 'Hi there! How are you doing today?

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.thinking: 'thinking'>, text='The human is asking me to say hi, which is a simple and friendly greeting. I should respond in a warm and welcoming way.', data={'citations': []}), Part(type=<PartType.text: 'text'>, text='Hi there! How are you doing today?', data={'citations': []})], data=None)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

🧠

I'll calculate both additions for you using

 the simple_add tool in parallel:

In [ ]:
comp.tool_calls

[ToolCall(id='toolu_01LyH4DdFWEbU9G7cg2XarVf', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_01JrpzizGDyz3BP2Ec2j1Ye1', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

In [ ]:
o.tool_calls

[ToolCall(id='toolu_013URHhgaocKHxi4ZHhbuCbV', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_01S2LXSsYWNLok8U9MRSD1az', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

In [ ]:
comp.message.content

[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to calculate two addition problems: 3 + 5 and 10 + 20. They want me to use the simple_add tool and they specifically mentioned "in parallel", which means I should make both function calls at the same time.\n\nLooking at the simple_add function, it takes two required parameters:\n- a (integer)\n- b (integer)\n\nFor the first calculation (3 + 5):\n- a = 3\n- b = 5\n\nFor the second calculation (10 + 20):\n- a = 10  \n- b = 20\n\nI have all the required parameters, so I can proceed with both function calls in parallel.', data={'type': 'thinking', 'thinking': 'The user is asking me to calculate two addition problems: 3 + 5 and 10 + 20. They want me to use the simple_add tool and they specifically mentioned "in parallel", which means I should make both function calls at the same time.\n\nLooking at the simple_add function, it takes two required parameters:\n- a (integer)\n- b (integer)\n\nFor the first calculation (3 + 

In [ ]:
o.message.content

[Part(type=<PartType.thinking: 'thinking'>, text="The user wants me to calculate 3 + 5 and 10 + 20 using the simple_add tool in parallel. I can make two function calls to the simple_add function with the respective parameters.\n\nFor 3 + 5: a = 3, b = 5\nFor 10 + 20: a = 10, b = 20\n\nI'll make both calls in parallel within the same function_calls block.", data={'citations': []}),
 Part(type=<PartType.text: 'text'>, text="I'll calculate both additions for you using the simple_add tool in parallel:", data={'citations': []}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_013URHhgaocKHxi4ZHhbuCbV', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'caller': {'type': 'direct'}}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01S2LXSsYWNLok8U9MRSD1az', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'caller': {'type': 'direct'}})]

In [ ]:
test_eq(comp.tool_calls[0].server, False)
for tc in comp.tool_calls: tc.id = '123'
for tc in o.tool_calls:    tc.id = '123'
test_eq(comp.tool_calls, o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=447, completion_tokens=300, total_tokens=747, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 300, 'service_tier': 'standard', 'inference_geo': 'not_available'}),
 Usage(prompt_tokens=447, completion_tokens=247, total_tokens=694, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 247}))

#### Web search

In [ ]:
def add_ant_link(citations):
    links = []
    for c in citations:
        if 'title' not in c: return ''
        title = c['title'].replace('"', '\\"')
        links.append(f'[*]({c["url"]} "{title}")')
    return ' '.join(links)

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            if citations:= o.get('citations'): print(add_ant_link(citations),end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Use web search to get the weather in Istanbul?"}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools=[{"type": "web_search_20250305", "name": "web_search"}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

Based

 on the current weather information for Istanbul, here's what I found:

**Current Weather in Istanbul (

...

The temperature for Wednesday is

 forecast to drop to 52°F

.

The weather appears to be transitional with pleasant

 temperatures today but rain expected tonight and tomorrow. Due

 to the poor air quality, those sensitive to pollution may want to limit outdoor activities.

No collation done in the non-streaming path, response parts are returned as is:

In [ ]:
len(comp.message.content)

18

Citations are attached to each `Part.data`:

In [ ]:
text_cites = L(comp.message.content).map(lambda p: (p.text, p.data.get('citations'))).filter(lambda o:o[1]); text_cites[:3]

[('Today (Tuesday, April 21) shows intervals of clouds and sunshine with a high of 70°F and a current temperature of 66°F', [{'type': 'web_search_result_location', 'cited_text': 'Istanbul, Istanbul Weather Today ... · Tue, Apr 21 · Intervals of clouds and sunshine Hi: 70° · Tonight: Overcast with a little rain late Lo: 49° · 3:...', 'url': 'https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251', 'title': 'Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather', 'encrypted_index': 'EpEBCioIDhgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDAbfLxIQbS6yY05UgBoM9jDNIpN1AY5YSe6LIjC2CD6rNf0Txl2pfi0rP+fggwJIWCpASWdIJd/pYMQoLydjy0QxU7glngkz95ayqU4qFUWskE3sEVAnXfy06l3sKbX2uL5UFxgE'}]), ('The "RealFeel" temperature is 67° with mostly sunny conditions', [{'type': 'web_search_result_location', 'cited_text': 'Istanbul, Istanbul Weather Today WinterCast Local {stormName} Tracker Hourly 10-Day Radar MinuteCast® Monthly Air Quality Health & Activities · News ·...', 'url':

In [ ]:
t, c = text_cites[0][0], text_cites[0][1][0]; c

{'type': 'web_search_result_location',
 'cited_text': 'Istanbul, Istanbul Weather Today ... · Tue, Apr 21 · Intervals of clouds and sunshine Hi: 70° · Tonight: Overcast with a little rain late Lo: 49° · 3:...',
 'url': 'https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251',
 'title': 'Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather',
 'encrypted_index': 'EpEBCioIDhgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDAbfLxIQbS6yY05UgBoM9jDNIpN1AY5YSe6LIjC2CD6rNf0Txl2pfi0rP+fggwJIWCpASWdIJd/pYMQoLydjy0QxU7glngkz95ayqU4qFUWskE3sEVAnXfy06l3sKbX2uL5UFxgE'}

In [ ]:
title = c['title'].replace('"', '\\"')
Markdown(f'[*]({c["url"]} "{title}") {t}')

<div class="prose">

[*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather") Today (Tuesday, April 21) shows intervals of clouds and sunshine with a high of 70°F and a current temperature of 66°F

</div>

Same with streaming:

In [ ]:
L(comp.message.content).attrgot('type'), L(o.message.content).attrgot('type')

([<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>],
 [<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>])

In [ ]:
text_cites = L(o.message.content).map(lambda p: (p.text, p.data.get('citations'))).filter(lambda o:o[1]); text_cites[:3]

[('Istanbul is experiencing intervals of clouds and sunshine with a high of 70°F today', [{'type': 'web_search_result_location', 'cited_text': 'Istanbul, Istanbul Weather Today ... · Tue, Apr 21 · Intervals of clouds and sunshine Hi: 70° · Tonight: Overcast with a little rain late Lo: 49° · 3:...', 'url': 'https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251', 'title': 'Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather', 'encrypted_index': 'Eo8BCioIDhgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDM2YlDUp+WGAjTMvXhoMelSCtairrqod04qEIjA1uUc+xF+RJg0+160h0UTkqmJhffXc4svVXbaGwHTFFL4Th9tHn8mnjlDcCEBJQogqE9itlpSbk1GLFuGksfAoK/0n3V4YBA=='}]), ('The current temperature is 66°F at 3:26 PM with a RealFeel® temperature of 67°', [{'type': 'web_search_result_location', 'cited_text': 'Istanbul, Istanbul Weather Today WinterCast Local {stormName} Tracker Hourly 10-Day Radar MinuteCast® Monthly Air Quality Health & Activities · News ·...', 'url': 'https://www.accuw

In [ ]:
t, c = text_cites[0][0], text_cites[0][1][0]; c

{'type': 'web_search_result_location',
 'cited_text': 'Istanbul, Istanbul Weather Today ... · Tue, Apr 21 · Intervals of clouds and sunshine Hi: 70° · Tonight: Overcast with a little rain late Lo: 49° · 3:...',
 'url': 'https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251',
 'title': 'Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather',
 'encrypted_index': 'Eo8BCioIDhgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDM2YlDUp+WGAjTMvXhoMelSCtairrqod04qEIjA1uUc+xF+RJg0+160h0UTkqmJhffXc4svVXbaGwHTFFL4Th9tHn8mnjlDcCEBJQogqE9itlpSbk1GLFuGksfAoK/0n3V4YBA=='}

In [ ]:
title = c['title'].replace('"', '\\"')
Markdown(f'[*]({c["url"]} "{title}") {t}')

<div class="prose">

[*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather") Istanbul is experiencing intervals of clouds and sunshine with a high of 70°F today

</div>

In [ ]:
test_eq(comp.tool_calls[0].server, True)
for tc in comp.tool_calls: tc.id = '123'
for tc in o.tool_calls:    tc.id = '123'
test_eq(comp.tool_calls, o.tool_calls)

In [ ]:
test_eq('server_tool_result' in L(comp.message.content).attrgot('type'), True)
test_eq('server_tool_result' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=12194, completion_tokens=472, total_tokens=12666, raw={'input_tokens': 12194, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 472, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}),
 Usage(prompt_tokens=12191, completion_tokens=449, total_tokens=12640, raw={'input_tokens': 12191, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 449, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}))

### Gemini Generate Events

In [ ]:
#| export
def normalize_gemini_event(ev, **kwargs):
    "Normalize Gemini stream event into Delta."
    cand = nested_idx(ev, 'candidates', 0) or {}
    finish_reason = canon_finish(cand.get("finishReason"), 'gemini')
    parts = nested_idx(cand, 'content', 'parts') or []
    thinking = "".join(p.get("text","") for p in parts if p.get("thought") and "text" in p)
    txt = "".join(p.get("text","") for p in parts if not p.get("thought") and "text" in p)
    tcs = gemini_tool_calls(ev)
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=txt, thinking=thinking, tool_calls=tcs, citations=listify(cand.get('groundingMetadata', [])), finish_reason=finish_reason, usage=usage_from_gemini(ev), raw=ev, **kwargs)

In [ ]:
#| export
def gemini_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if not (last_typ or last_idx): return 0,0
    return last_idx + 1, last_idx + 1

In [ ]:
#| export
_acollect_stream_gemini = partial(_acollect_stream, index_fn=gemini_index_fn, api_name=ApiName.gemini)

#### Text

In [ ]:
mn,inp = 'models/gemini-3-flash-preview',[{"role": "user", "parts": [{"text": "Hi how are you?"}]}]
resp = await gem_cli.models.generate_content(model=mn,contents=inp)
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

I'm doing well, thank you for asking! How are you doing

 today? Is there anything I can help you with?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text="I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?", data={'text': "I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?", 'thoughtSignature': 'EtsDCtgDAQw51sduOJlt+bXK6LX8RRx6a06/V+iQMSCkUnWvi8wyLTR6U0Z+cwkaNnOkDgQz4ijJ5efcKRCP/7hx8iIwlZRJxWIljV20qlADtWgVZFttXMmRSJbSQJrG6L3sc4axuPwdkyZEIUcnX8kNXgCeAlTWpU2Uaami4DSEctIDbOexPk1BTHvYcwv6Pi2sHW1TbSjmcczgYl7CF81iI3uzFAETcxA1mNtiWqZKn17pDbJXwfq+t+a5cVplrRIE4/piUndtB51c7SSZymBUTKiGwCLHaRDihnQZdpNpJ1RN45r+HCSTeL069hpjVlwDMiE7S3DiWFVFaGMqsupU6fO7dTSx/EDfOhlXKEGg1NVjyq/nKAwClzfMfbzTNyhxm8hjFsCN1HZIO34v+rNdk/YeX/+rkfzocBU/qxo3Tv7EdBshzu6HyTxpygTcMHIMK03S/10avgvIeJnOXcE3/y7jGwZHDLPRd82gfWJXno0T5G7Gpz4cYJSK+jYz7NNpohxXNIMJgKtEDHWp/EcSnBaqmHaE/rXcBqsi6bhKzTEk6ty+W9fQT3O9yDMd14nqUCCKn7ca+eVyVFrexkhiPD6esyFQlksq2xdIhopJ21XqrLqYair+PqlplQ=='})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text="I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?", data={'citations': []})], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=6, completion_tokens=26, total_tokens=144, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 144, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 112}),
 Usage(prompt_tokens=6, completion_tokens=26, total_tokens=134, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 134, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 102}))

#### Thinking

- **Thinking always happens internally** for Gemini 3/2.5 models — token cost visible in `thoughtsTokenCount` in usage metadata
- **Thought text in response** only returned when `include_thoughts: True` is explicitly set in `generation_config.thinking_config` — defaults to `false`
- **`thoughtSignature`** is always returned regardless of `includeThoughts` — required for multi-turn context continuity. Missing signatures in function calling result in a 400 error; for text/chat, omitting them degrades reasoning quality
- **`thinking_level`** controls reasoning depth: `high` (default for Gemini 3), `medium`, `low`, `minimal` (Flash only)

In [ ]:
mn,inp = 'models/gemini-3-flash-preview',[{"role": "user", "parts": [{"text": "What is 31231231 * 12312?"}]}]
resp = await gem_cli.models.generate_content(model=mn,contents=inp,generation_config={"thinking_config": {"include_thoughts": True}})
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

🧠

31,231,231 * 12,312 = **

384,518,916,072**

In [ ]:
L(comp.message.content).attrgot('type'), #comp.message.content

([<PartType.thinking: 'thinking'>, <PartType.text: 'text'>],)

In [ ]:
L(o.message.content).attrgot('type')#, o.message.content

[<PartType.thinking: 'thinking'>, <PartType.text: 'text'>]

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
L(comp.message.content).attrgot('type'), L(o.message.content).attrgot('type')

([<PartType.thinking: 'thinking'>, <PartType.text: 'text'>],
 [<PartType.thinking: 'thinking'>, <PartType.text: 'text'>])

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=20, completion_tokens=460, total_tokens=1970, raw={'promptTokenCount': 20, 'candidatesTokenCount': 460, 'totalTokenCount': 1970, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1490}),
 Usage(prompt_tokens=20, completion_tokens=36, total_tokens=802, raw={'promptTokenCount': 20, 'candidatesTokenCount': 36, 'totalTokenCount': 802, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 746}))

#### Tool call

In [ ]:
inp = [{"role": "user", "parts": [{"text": 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=[{"functionDeclarations": [sch['function']]}])
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='tthvryw0', name='simple_add', arguments={'b': 5, 'a': 3}, server=False, extra={'thoughtSignature': 'EsoCCscCAQw51sfVNL7zEAoaXAB2Erd4oCqUm2HY1QEIkyJXYGLB8aLK5SzFqcMxY+i/mUX7FovVAWhFSAwfYGhfNjXakFTIqy151LK8QtP9judJ2TqzjoDKfKIVv+9oLJLnfIsnQpoiWTuRqsfvPh4DqTo/GuzNo5fg6UvjG8KvZrEOhPUAJjXlQG+qgXK1zjbY5tCAnDej9e2H0/+CpolI6XX3uTSaRPkfn90rrWuXUPkaMoumRsbBOzhnBHmh3X8d29lOn7MrWkWwf1m7U6Z+KFheFDu1XQJka7r8+qGJKorzhrZI1qI4DH/pRweDycEJVq1ZML5kQNzvZ7d4BEe+kF803P7LO7EYgIHQCsKQWAm3TnApGp3RvYv3u8H3J+qsUljrMlFVNi72EL12DdplINQadJeXsyDYjOoUdBSe2vKi8kz3rGR+VsVZ'}),
 ToolCall(id='e46t5lub', name='simple_add', arguments={'b': 20, 'a': 10}, server=False, extra={})]

In [ ]:
o.tool_calls

[ToolCall(id='7pdbed70', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'thoughtSignature': 'EpQDCpEDAQw51sfNUgJWlp41jx084xQ3IAJm1xgHlxrz2d8C1yelJXTa8+mrhMAA9xiQ/O/ow88noMXzafig+n8FcJFuY9Wsws+OJYm9YSxtxgYGbtlmlFgr5I9rKLCCIXKtiSY0uiZBLftJ0vj0K99GMRmCKROeRpHPlXddCQJz1zfC7csoBjhycbIe8fI/xYszQC56Os+oRw30lG0Zk3LVkhGb1rtKuPIrFZGAdMGNwe+zsfXzR4WozRQW5+W1Oo5emwP0atlU3ctfhDbR16FODDBSit3sacH+/PkpJ2rKA7Z4CT5kFvs0Ztnr7r72r98+KbSKSi18FaRDKZhoWkPL0SUQ2CangayizD+AoJrjKJuoMN8Lrc7XXQWvNHipdRilLfx6e1ji7RndoBW1Z6BkDgbjhFHKUxHtW5eS5jz/J12CuurhAEyqRAq87MNXo5hr1CFBEtwSWr2HnBRvB2WfSVzwiCAzMyN1BZvDfcEpjhgApmUQdGVjNIWDUMf8pomALHaOzyDD3SYxkvqAm+4qIZHNIio='}),
 ToolCall(id='l74gmgxz', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={})]

In [ ]:
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'tthvryw0', 'name': 'simple_add', 'arguments': {'b': 5, 'a': 3}, 'server': False, 'thoughtSignature': 'EsoCCscCAQw51sfVNL7zEAoaXAB2Erd4oCqUm2HY1QEIkyJXYGLB8aLK5SzFqcMxY+i/mUX7FovVAWhFSAwfYGhfNjXakFTIqy151LK8QtP9judJ2TqzjoDKfKIVv+9oLJLnfIsnQpoiWTuRqsfvPh4DqTo/GuzNo5fg6UvjG8KvZrEOhPUAJjXlQG+qgXK1zjbY5tCAnDej9e2H0/+CpolI6XX3uTSaRPkfn90rrWuXUPkaMoumRsbBOzhnBHmh3X8d29lOn7MrWkWwf1m7U6Z+KFheFDu1XQJka7r8+qGJKorzhrZI1qI4DH/pRweDycEJVq1ZML5kQNzvZ7d4BEe+kF803P7LO7EYgIHQCsKQWAm3TnApGp3RvYv3u8H3J+qsUljrMlFVNi72EL12DdplINQadJeXsyDYjOoUdBSe2vKi8kz3rGR+VsVZ'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'e46t5lub', 'name': 'simple_add', 'arguments': {'b': 20, 'a': 10}, 'server': False})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': '7pdbed70', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'thoughtSignature': 'EpQDCpEDAQw51sfNUgJWlp41jx084xQ3IAJm1xgHlxrz2d8C1yelJXTa8+mrhMAA9xiQ/O/ow88noMXzafig+n8FcJFuY9Wsws+OJYm9YSxtxgYGbtlmlFgr5I9rKLCCIXKtiSY0uiZBLftJ0vj0K99GMRmCKROeRpHPlXddCQJz1zfC7csoBjhycbIe8fI/xYszQC56Os+oRw30lG0Zk3LVkhGb1rtKuPIrFZGAdMGNwe+zsfXzR4WozRQW5+W1Oo5emwP0atlU3ctfhDbR16FODDBSit3sacH+/PkpJ2rKA7Z4CT5kFvs0Ztnr7r72r98+KbSKSi18FaRDKZhoWkPL0SUQ2CangayizD+AoJrjKJuoMN8Lrc7XXQWvNHipdRilLfx6e1ji7RndoBW1Z6BkDgbjhFHKUxHtW5eS5jz/J12CuurhAEyqRAq87MNXo5hr1CFBEtwSWr2HnBRvB2WfSVzwiCAzMyN1BZvDfcEpjhgApmUQdGVjNIWDUMf8pomALHaOzyDD3SYxkvqAm+4qIZHNIio='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'l74gmgxz', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=77, completion_tokens=38, total_tokens=206, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 206, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 91}),
 Usage(prompt_tokens=77, completion_tokens=38, total_tokens=203, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 203, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 88}))

#### Web search

Gemini's Google Search is a transparent server-side tool — unlike Anthropic/OpenAI, it produces no `functionCall` parts or tool_use blocks. Search results appear as `groundingMetadata` on the candidate, detected by `usage_from_gemini` to set `server_tool_use: {"google_search": 1}`. The response content is plain text only, so `tool_calls` is always `[]`.

In [ ]:
inp = [{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}])
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

Today in **Istanbul** (Monday, April 20, 202

6), the weather is **sunny and mild**.

Here are the details for today's forecast:
*   **Temperature

:** High of **17°C (62°F)** and a low of **8°C (46

°C)**.
*   **Conditions:** The day is primarily sunny, turning clear with periodic clouds tonight.
*   

**Rain:** There is a 0% chance of rain during the day, increasing slightly to 5% overnight.
*   

**Humidity:** Around 38–52%.

**Looking Ahead:**
If you are planning for the next few days

, keep an umbrella handy. A new weather system is expected to arrive tomorrow, **Tuesday, April 21**, bringing

 light rain and cooler temperatures (around 13°C) by midweek.

In [ ]:
len(comp.message.content)

1

In [ ]:
t,c = comp.message.content[0].text, comp.message.content[0].data['citations']

In [ ]:
def add_gem_links(t, c):
    for s in c['groundingSupports']:
        links = ', '.join(f'<a href="{c["groundingChunks"][i]["web"]["uri"]}">{i+1}</a>' for i in s['groundingChunkIndices'])
        t = t.replace(s['segment']['text'], f"{s['segment']['text']} {links}", 1)
    return t

In [ ]:
Markdown(add_gem_links(t,c))

<div class="prose">

Today in Istanbul, the weather is currently **cloudy** with a temperature of **17°C (63°F)**. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGQnrTNJXKyeGXnwDYHCKjKJ59KTgzFbncRT17rsK9Ei27kdRBsHK8kmwhq69gx3QIZSvX3V13dtWZlUnEo9LSbETgccRlTuonQOkJE8fb5GoDu2sBJQ9Ht6yRyMZy6xb5ep1iAPouOkHeWbFPuESJwZdY=">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5tkGa4hmu9X-J8FAEyurhAM3io6t5CbWI_Op4yfKLsEJtSq-NyHXuA9eC95U11PWw786BOS_Fzn-0El-1Wg54uqu2nuJcOHzgo0wHAkFV0X2zOrGfssbzyKXiXUOrmhvMj05y8dn9gju4RlO2k_m26koYARHhhzSfja9R12tFHg==">2</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH3EFnGKM70D46aKF8GmOBWGKlfFQM5sOIAqH4Y1Eku6aH_pBQCbuoE2UmIHV1ksLCj0JFwM7u6Z7pndRXJTjDCP1PU2HaLIkEeBjsqFNjcFtA8ZGbHXBmByuC9cPtRvG90lq40Oy8JY5uZtuZtk1xBKH1t996N7bb9pyL4r70K5w==">3</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">4</a> 

The forecast for the rest of the day and tonight includes:
*   **Conditions:** Overcast with a chance of light rain developing, particularly this evening (about a 55% chance tonight). <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGQnrTNJXKyeGXnwDYHCKjKJ59KTgzFbncRT17rsK9Ei27kdRBsHK8kmwhq69gx3QIZSvX3V13dtWZlUnEo9LSbETgccRlTuonQOkJE8fb5GoDu2sBJQ9Ht6yRyMZy6xb5ep1iAPouOkHeWbFPuESJwZdY=">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5tkGa4hmu9X-J8FAEyurhAM3io6t5CbWI_Op4yfKLsEJtSq-NyHXuA9eC95U11PWw786BOS_Fzn-0El-1Wg54uqu2nuJcOHzgo0wHAkFV0X2zOrGfssbzyKXiXUOrmhvMj05y8dn9gju4RlO2k_m26koYARHhhzSfja9R12tFHg==">2</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH3EFnGKM70D46aKF8GmOBWGKlfFQM5sOIAqH4Y1Eku6aH_pBQCbuoE2UmIHV1ksLCj0JFwM7u6Z7pndRXJTjDCP1PU2HaLIkEeBjsqFNjcFtA8ZGbHXBmByuC9cPtRvG90lq40Oy8JY5uZtuZtk1xBKH1t996N7bb9pyL4r70K5w==">3</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">4</a>
*   **Temperature Range:** A daytime high of **18°C (64°F)** and an overnight low of **10°C (50°F)**. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5tkGa4hmu9X-J8FAEyurhAM3io6t5CbWI_Op4yfKLsEJtSq-NyHXuA9eC95U11PWw786BOS_Fzn-0El-1Wg54uqu2nuJcOHzgo0wHAkFV0X2zOrGfssbzyKXiXUOrmhvMj05y8dn9gju4RlO2k_m26koYARHhhzSfja9R12tFHg==">2</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH3EFnGKM70D46aKF8GmOBWGKlfFQM5sOIAqH4Y1Eku6aH_pBQCbuoE2UmIHV1ksLCj0JFwM7u6Z7pndRXJTjDCP1PU2HaLIkEeBjsqFNjcFtA8ZGbHXBmByuC9cPtRvG90lq40Oy8JY5uZtuZtk1xBKH1t996N7bb9pyL4r70K5w==">3</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGQnrTNJXKyeGXnwDYHCKjKJ59KTgzFbncRT17rsK9Ei27kdRBsHK8kmwhq69gx3QIZSvX3V13dtWZlUnEo9LSbETgccRlTuonQOkJE8fb5GoDu2sBJQ9Ht6yRyMZy6xb5ep1iAPouOkHeWbFPuESJwZdY=">1</a>
*   **Humidity:** Around **52%–62%**. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGQnrTNJXKyeGXnwDYHCKjKJ59KTgzFbncRT17rsK9Ei27kdRBsHK8kmwhq69gx3QIZSvX3V13dtWZlUnEo9LSbETgccRlTuonQOkJE8fb5GoDu2sBJQ9Ht6yRyMZy6xb5ep1iAPouOkHeWbFPuESJwZdY=">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5tkGa4hmu9X-J8FAEyurhAM3io6t5CbWI_Op4yfKLsEJtSq-NyHXuA9eC95U11PWw786BOS_Fzn-0El-1Wg54uqu2nuJcOHzgo0wHAkFV0X2zOrGfssbzyKXiXUOrmhvMj05y8dn9gju4RlO2k_m26koYARHhhzSfja9R12tFHg==">2</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH3EFnGKM70D46aKF8GmOBWGKlfFQM5sOIAqH4Y1Eku6aH_pBQCbuoE2UmIHV1ksLCj0JFwM7u6Z7pndRXJTjDCP1PU2HaLIkEeBjsqFNjcFtA8ZGbHXBmByuC9cPtRvG90lq40Oy8JY5uZtuZtk1xBKH1t996N7bb9pyL4r70K5w==">3</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">4</a>
*   **Wind:** Generally light to moderate. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGQnrTNJXKyeGXnwDYHCKjKJ59KTgzFbncRT17rsK9Ei27kdRBsHK8kmwhq69gx3QIZSvX3V13dtWZlUnEo9LSbETgccRlTuonQOkJE8fb5GoDu2sBJQ9Ht6yRyMZy6xb5ep1iAPouOkHeWbFPuESJwZdY=">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5tkGa4hmu9X-J8FAEyurhAM3io6t5CbWI_Op4yfKLsEJtSq-NyHXuA9eC95U11PWw786BOS_Fzn-0El-1Wg54uqu2nuJcOHzgo0wHAkFV0X2zOrGfssbzyKXiXUOrmhvMj05y8dn9gju4RlO2k_m26koYARHhhzSfja9R12tFHg==">2</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH3EFnGKM70D46aKF8GmOBWGKlfFQM5sOIAqH4Y1Eku6aH_pBQCbuoE2UmIHV1ksLCj0JFwM7u6Z7pndRXJTjDCP1PU2HaLIkEeBjsqFNjcFtA8ZGbHXBmByuC9cPtRvG90lq40Oy8JY5uZtuZtk1xBKH1t996N7bb9pyL4r70K5w==">3</a>

If you are heading out this evening, it is a good idea to carry an umbrella as the likelihood of rain increases later in the day. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5tkGa4hmu9X-J8FAEyurhAM3io6t5CbWI_Op4yfKLsEJtSq-NyHXuA9eC95U11PWw786BOS_Fzn-0El-1Wg54uqu2nuJcOHzgo0wHAkFV0X2zOrGfssbzyKXiXUOrmhvMj05y8dn9gju4RlO2k_m26koYARHhhzSfja9R12tFHg==">2</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH3EFnGKM70D46aKF8GmOBWGKlfFQM5sOIAqH4Y1Eku6aH_pBQCbuoE2UmIHV1ksLCj0JFwM7u6Z7pndRXJTjDCP1PU2HaLIkEeBjsqFNjcFtA8ZGbHXBmByuC9cPtRvG90lq40Oy8JY5uZtuZtk1xBKH1t996N7bb9pyL4r70K5w==">3</a>

</div>

In [ ]:
len(o.message.content)

1

In [ ]:
L(o.message.content).attrgot('type')

[<PartType.text: 'text'>]

In [ ]:
t,c = o.message.content[0].text, o.message.content[0].data['citations'][0]

In [ ]:
Markdown(add_gem_links(t,c))

<div class="prose">

Today in **Istanbul** (Monday, April 20, 2026), the weather is **sunny and mild**. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEo3BuFaZGW6uGDEpHX0D2up2BPGsrrULE89jzXQDlZWjWDvxWK-YHIavEgAMM_UC2aV6l9EwE6C5AoXpCt1Ivu2niGkJFJNYX6j19VoTg_dWEQ0BQQbkftPypHn9dwGwQRyKF1DU6DC82MkP5uNnSc-PEu">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEbdhkN-llyVn0h6sxXck8KGa1Z5rv_B7Wzbx1WC49FPFMq9kez0jBDh7kI1bs9z_dDA0h6OISwtEDWQolzMfUCI-h-OIJaTJz45KwyI1jxSXJf21N6ADphRV2bSwHyid6O-zPaE2u52wQSqlBkbsme5E0mgVS5zATkIwNnTmp0nx1y">2</a>

Here are the details for today's forecast:
*   **Temperature:** High of **17°C (62°F)** and a low of **8°C (46°C)**. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEo3BuFaZGW6uGDEpHX0D2up2BPGsrrULE89jzXQDlZWjWDvxWK-YHIavEgAMM_UC2aV6l9EwE6C5AoXpCt1Ivu2niGkJFJNYX6j19VoTg_dWEQ0BQQbkftPypHn9dwGwQRyKF1DU6DC82MkP5uNnSc-PEu">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHn5GZ516SxG8PbzuutMB12asipC_yIggcjwNz8w5NGjQ2a4UP_D9-f2KPfRd4qbyIQRs39QAV4QCwIg4whglfaSY-1xYXt9GYo8LV6mBy_H9WcUpFeZOveiouh497CvkJ2CG1GhEtJSWWPCMaux2cuT6Bv">3</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">4</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEbdhkN-llyVn0h6sxXck8KGa1Z5rv_B7Wzbx1WC49FPFMq9kez0jBDh7kI1bs9z_dDA0h6OISwtEDWQolzMfUCI-h-OIJaTJz45KwyI1jxSXJf21N6ADphRV2bSwHyid6O-zPaE2u52wQSqlBkbsme5E0mgVS5zATkIwNnTmp0nx1y">2</a>
*   **Conditions:** The day is primarily sunny, turning clear with periodic clouds tonight. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEo3BuFaZGW6uGDEpHX0D2up2BPGsrrULE89jzXQDlZWjWDvxWK-YHIavEgAMM_UC2aV6l9EwE6C5AoXpCt1Ivu2niGkJFJNYX6j19VoTg_dWEQ0BQQbkftPypHn9dwGwQRyKF1DU6DC82MkP5uNnSc-PEu">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHn5GZ516SxG8PbzuutMB12asipC_yIggcjwNz8w5NGjQ2a4UP_D9-f2KPfRd4qbyIQRs39QAV4QCwIg4whglfaSY-1xYXt9GYo8LV6mBy_H9WcUpFeZOveiouh497CvkJ2CG1GhEtJSWWPCMaux2cuT6Bv">3</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEbdhkN-llyVn0h6sxXck8KGa1Z5rv_B7Wzbx1WC49FPFMq9kez0jBDh7kI1bs9z_dDA0h6OISwtEDWQolzMfUCI-h-OIJaTJz45KwyI1jxSXJf21N6ADphRV2bSwHyid6O-zPaE2u52wQSqlBkbsme5E0mgVS5zATkIwNnTmp0nx1y">2</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">4</a>
*   **Rain:** There is a 0% chance of rain during the day, increasing slightly to 5% overnight. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEo3BuFaZGW6uGDEpHX0D2up2BPGsrrULE89jzXQDlZWjWDvxWK-YHIavEgAMM_UC2aV6l9EwE6C5AoXpCt1Ivu2niGkJFJNYX6j19VoTg_dWEQ0BQQbkftPypHn9dwGwQRyKF1DU6DC82MkP5uNnSc-PEu">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHn5GZ516SxG8PbzuutMB12asipC_yIggcjwNz8w5NGjQ2a4UP_D9-f2KPfRd4qbyIQRs39QAV4QCwIg4whglfaSY-1xYXt9GYo8LV6mBy_H9WcUpFeZOveiouh497CvkJ2CG1GhEtJSWWPCMaux2cuT6Bv">3</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEbdhkN-llyVn0h6sxXck8KGa1Z5rv_B7Wzbx1WC49FPFMq9kez0jBDh7kI1bs9z_dDA0h6OISwtEDWQolzMfUCI-h-OIJaTJz45KwyI1jxSXJf21N6ADphRV2bSwHyid6O-zPaE2u52wQSqlBkbsme5E0mgVS5zATkIwNnTmp0nx1y">2</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">4</a>
*   **Humidity:** Around 38–52%. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEo3BuFaZGW6uGDEpHX0D2up2BPGsrrULE89jzXQDlZWjWDvxWK-YHIavEgAMM_UC2aV6l9EwE6C5AoXpCt1Ivu2niGkJFJNYX6j19VoTg_dWEQ0BQQbkftPypHn9dwGwQRyKF1DU6DC82MkP5uNnSc-PEu">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHn5GZ516SxG8PbzuutMB12asipC_yIggcjwNz8w5NGjQ2a4UP_D9-f2KPfRd4qbyIQRs39QAV4QCwIg4whglfaSY-1xYXt9GYo8LV6mBy_H9WcUpFeZOveiouh497CvkJ2CG1GhEtJSWWPCMaux2cuT6Bv">3</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEbdhkN-llyVn0h6sxXck8KGa1Z5rv_B7Wzbx1WC49FPFMq9kez0jBDh7kI1bs9z_dDA0h6OISwtEDWQolzMfUCI-h-OIJaTJz45KwyI1jxSXJf21N6ADphRV2bSwHyid6O-zPaE2u52wQSqlBkbsme5E0mgVS5zATkIwNnTmp0nx1y">2</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">4</a>

**Looking Ahead:**
If you are planning for the next few days, keep an umbrella handy. A new weather system is expected to arrive tomorrow, **Tuesday, April 21**, bringing light rain and cooler temperatures (around 13°C) by midweek. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEo3BuFaZGW6uGDEpHX0D2up2BPGsrrULE89jzXQDlZWjWDvxWK-YHIavEgAMM_UC2aV6l9EwE6C5AoXpCt1Ivu2niGkJFJNYX6j19VoTg_dWEQ0BQQbkftPypHn9dwGwQRyKF1DU6DC82MkP5uNnSc-PEu">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEbdhkN-llyVn0h6sxXck8KGa1Z5rv_B7Wzbx1WC49FPFMq9kez0jBDh7kI1bs9z_dDA0h6OISwtEDWQolzMfUCI-h-OIJaTJz45KwyI1jxSXJf21N6ADphRV2bSwHyid6O-zPaE2u52wQSqlBkbsme5E0mgVS5zATkIwNnTmp0nx1y">2</a>

</div>

In [ ]:
comp.finish_reason, o.finish_reason

('stop', 'stop')

In [ ]:
comp.usage, o.usage

(Usage(prompt_tokens=67, completion_tokens=166, total_tokens=515, raw={'promptTokenCount': 67, 'candidatesTokenCount': 166, 'totalTokenCount': 515, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 67}], 'thoughtsTokenCount': 282, 'server_tool_use': {'google_search': 1}}),
 Usage(prompt_tokens=52, completion_tokens=188, total_tokens=388, raw={'promptTokenCount': 52, 'candidatesTokenCount': 188, 'totalTokenCount': 388, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 52}], 'thoughtsTokenCount': 148, 'server_tool_use': {'google_search': 1}}))

### acollect_stream

In [ ]:
#| export
async def acollect_stream(it, model=None, api_name=None, vendor_name=None):
    _norm = {ApiName.openai: normalize_openai_response_event, ApiName.openai_chat: normalize_openai_chat_delta,
             ApiName.anthropic: normalize_anthropic_event, ApiName.gemini: normalize_gemini_event}
    _coll = {ApiName.openai: _acollect_stream_openai_responses, ApiName.openai_chat: _acollect_stream_openai_chat,
             ApiName.anthropic: _acollect_stream_anthropic, ApiName.gemini: _acollect_stream_gemini}
    if api_name not in _norm: raise ValueError(f"Unknown api_name: {api_name}")
    async for o in _coll[api_name](norm_and_yield(it, _norm[api_name]), model=model, vendor_name=vendor_name): yield o

In [ ]:
# OpenAI Responses
mn, inp = 'gpt-4o-mini', 'Say hi'
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
async for o in acollect_stream(resp, model=mn, api_name=ApiName.openai): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi

 there

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
# OpenAI Chat
mn, msgs = 'gpt-4o-mini', [{"role": "user", "content": "Say hi"}]
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=msgs, stream=True, stream_options={"include_usage": True})
async for o in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
# Anthropic
mn, msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Say hi"}]
hdrs = {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, max_tokens=100, headers=hdrs, stream=True)
async for o in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi! How are

 you doing today?

In [ ]:
# Gemini
mn, inp = 'models/gemini-3-flash-preview', [{"role": "user", "parts": [{"text": "Say hi"}]}]
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp, model=mn, api_name=ApiName.gemini): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi! How can I help you today?

In [ ]:
o.api_name, o.vendor_name

(<api_name.gemini: 'gemini'>, None)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()